In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

In [29]:
#Dynamic search path configuration
import sys
import os

#Adjusting search path based on notebook's depth in directory --> iterates through directories until it finds directory name: US_conflict_forecasting
current_dir = os.getcwd()
while 'US_conflict_forecasting' not in os.path.basename(current_dir):
    current_dir = os.path.dirname(current_dir)

#Adjusting search path based on 'new' current directory
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from project_setup import setup_project
project_root_dir = setup_project()

import pandas as pd

In [44]:
# Load the data

data = pd.read_csv(os.path.join(project_root_dir, '2_Dataset/d_Final_Dataset/Final_Dataset_noLag.csv'))


In [45]:
# Drop some unnecessary columns
data = data.drop(columns = ['month_news', 'admin2', 'Post Office City Name'])

In [46]:
# Recode urban variable to categorical
data['Rural_Urban_Commerical Status'] = data['Rural_Urban_Commerical Status'].astype('category')

categories = ['Rural', 'Urban', 'Commerical']  # Specify your categories if you know them

# Initialize the LabelEncoder
label_encoder = LabelEncoder()

# Fit the LabelEncoder to your categories
label_encoder.fit(categories)

# Transform your variable to encoded values
data['Rural_Urban_Commerical Status'] = label_encoder.transform(data['Rural_Urban_Commerical Status'])

In [47]:
# Count missing values
nan_count = data.isna().sum().sum()
print("Number of NaN values:", nan_count)

Number of NaN values: 2088


In [34]:
'''
def show_missing_values(data):
    for name, amount in data.items():
        print(f"Missing values in the {name.capitalize()} dataset:")
        print(amount.isnull().sum())
        print('===' * 18)
        print()
'''

'\ndef show_missing_values(data):\n    for name, amount in data.items():\n        print(f"Missing values in the {name.capitalize()} dataset:")\n        print(amount.isnull().sum())\n        print(\'===\' * 18)\n        print()\n'

In [48]:
# Find and count values that indicate '-'
def count_dash(series):
    return series.eq('-').sum()

count_dash_data = data.applymap(lambda x: x == '-').sum().sum()

print("Number of values with '-' in the entire DataFrame:", count_dash_data)

C:\Users\danie\AppData\Local\Temp\ipykernel_11996\1411631116.py:5: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  count_dash_data = data.applymap(lambda x: x == '-').sum().sum()


Number of values with '-' in the entire DataFrame: 12372


In [49]:
# Find columns that hold '-'
columns_with_dash = data.apply(lambda x: '-' in x.values)

# Save those columns to list
columns_with_dash = columns_with_dash[columns_with_dash].index.tolist()

# It's median / mean columns that are calculated from other variables

In [50]:
data['month'] = data['month'].apply(lambda x: int(str(x)[-2:]))

In [51]:
# Replace each '-' with NaN, so can be imputed later
data.replace('-', np.nan, inplace=True)

In [52]:
# Looking for columns with the string '250,000+'
columns_with_string = data.applymap(lambda x: isinstance(x, str) and '250,000+' in x).any()

# Save those columns to a list
columns_containing_string = columns_with_string[columns_with_string].index.tolist()

# Inspect
print("Columns containing '250,000+':", columns_containing_string)

C:\Users\danie\AppData\Local\Temp\ipykernel_11996\1338546616.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  columns_with_string = data.applymap(lambda x: isinstance(x, str) and '250,000+' in x).any()


Columns containing '250,000+': ['Median Household Income, 2022', 'Median Family Income, 2022']


In [53]:
# Replace '250,000+' with '250,000' in the specified columns
for column in columns_containing_string:
    data[column] = data[column].str.replace('250,000+', '250,000')

In [54]:
# Select any column that is non-numeric
non_numeric_columns = data.select_dtypes(exclude=['number']).columns.tolist()

# Transform the non-numeric columns to numeric

for column in non_numeric_columns:
    data[column] = pd.to_numeric(data[column], errors='coerce')

In [55]:
# Imputing missing values

imputer = SimpleImputer(missing_values=np.nan, strategy='mean')

# Fit the imputer on the data and transform the column in one step
data_imputed = imputer.fit_transform(data)


data_imputed = pd.DataFrame(data_imputed, columns=data.columns)

# Back to df
# data = pd.DataFrame(data)

In [56]:
# Save the final dataset to a CSV file
output_directory_1 = os.path.join(project_root_dir, '2_Dataset/d_Final_Dataset')
output_filename_1 = "Final_Dataset_noLag_clean.csv"
output_path_1 = os.path.join(output_directory_1, output_filename_1)

data_imputed.to_csv(output_path_1, index=False)